In [2]:
%pip install datasets

   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   --------------------------------------- 604.4/604.4 kB 10.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/28.5 MB ? eta -:--:--
   -- ------------------------------------- 2.1/28.5 MB 10.7 MB/s eta 0:00:03
   ------ --------------------------------- 4.7/28.5 MB 11.4 MB/s eta 0:00:03
   --------- ------------------------------ 7.1/28.5 MB 11.5 MB/s eta 0:00:02
   ------------- -------------------------- 9.7/28.5 MB 11.4 MB/s eta 0:00:02
   ---------------- ----------------------- 12.1/28.5 MB 11.5 MB/s eta 0:00:02
   -------------------- ------------------- 14.4/28.5 MB 11.6 MB/s eta 0:00:02
   ----------------------- ---------------- 17.0/28.5 MB 11.6 MB/s eta 0:00:01
   --------------------------- ------------ 19.4/28.5 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 21.8/28.5 MB 11.7 MB/s eta 0:00:01
   ---------------------------------- ----- 24.4/28.5 MB 11.7 MB/s eta 0


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
import pandas as pd

nifty = load_dataset("raeidsaqur/NIFTY")
print(nifty)

df = nifty["train"].to_pandas()
row = df.iloc[0]
print("date:", row["date"], "| label:", row["label"], "| pct_change:", row["pct_change"])
print("news type:", type(row["news"]))
print(str(row["news"])[:600])

for split in ["train", "valid", "test"]:
    print(split, nifty[split].to_pandas()["label"].value_counts().to_dict())

h:\Desktop\financial-news-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating valid split: 100%|██████████| 317/317 [00:00<00:00, 14824.66 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 1477
    })
    test: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 317
    })
    valid: Dataset({
        features: ['id', 'date', 'context', 'news', 'conversations', 'label', 'pct_change'],
        num_rows: 317
    })
})
date: 2010-01-06 00:00:00 | label: Neutral | pct_change: 0.0042
news type: <class 'str'>
China Officials Likely Knew of Bad Milk
Sony's CEO on Strategy in 3-D Technology
Copper Settles at 16-Month High
Gold Ends Near 3-Week High
Kraft Gets Antitrust Clearance
European Stocks Close Flat
M&S Sales Miss Expectations
Yen Boosts Japan's Exporters
Future Group: Value Retail Unit May Consider IPO
Dollar, Yen Gain on Haven Plays
Clothing Sales Sagged in December
Luxury Logos Draw Asian Shoppers
Mexican Stocks Have Record in Sight
Cold Snap Heats Up OJ Pric

In [4]:
%pip install yfinance
import yfinance as yf

spy = yf.download("SPY", start="2010-01-01", end="2010-01-20", auto_adjust=False, progress=False)
spy_change = spy["Close"]["SPY"].pct_change().round(4)
print("Real SPY daily change:\n", spy_change)

print("\nNIFTY rows:")
print(df[["date", "pct_change", "label"]].head(6).to_string(index=False))

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 11.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -------------------- ------------------- 2.1/4.0 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 10.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Real SPY daily change:
 Date
2010-01-04       NaN
2010-01-05    0.0026
2010-01-06    0.0007
2010-01-07    0.0042
2010-01-08    0.0033
2010-01-11    0.0014
2010-01-12   -0.0093
2010-01-13    0.0084
2010-01-14    0.0027
2010-01-15   -0.0112
2010-01-19    0.0125
Name: SPY, dtype: float64

NIFTY rows:
      date  pct_change   label
2010-01-06      0.0042 Neutral
2010-01-07      0.0033 Neutral
2010-01-11     -0.0093    Fall
2010-01-12      0.0084    Rise
2010-01-13      0.0027 Neutral
2010-01-14     -0.0112    Fall


In [5]:
import torch, numpy as np
from pathlib import Path
from transformers import BertTokenizer, BertForSequenceClassification

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FINBERT_DIR = PROJECT_ROOT / "models" / "finbert"
tokenizer = BertTokenizer.from_pretrained(FINBERT_DIR)
finbert = BertForSequenceClassification.from_pretrained(FINBERT_DIR).eval()

days = pd.concat([nifty[s].to_pandas().assign(split=s) for s in ["train", "valid", "test"]], ignore_index=True)
days["headlines"] = days["news"].str.split("\n").apply(lambda hs: [h.strip() for h in hs if h.strip()])
print("days:", len(days), "| total headlines:", days["headlines"].str.len().sum())

def day_features(headlines):
    probs = []
    for i in range(0, len(headlines), 64):
        enc = tokenizer(headlines[i:i+64], truncation=True, max_length=64, padding=True, return_tensors="pt")
        with torch.no_grad():
            probs.append(finbert(**enc).logits.softmax(dim=-1).numpy())
    p = np.vstack(probs)                     # columns: negative, neutral, positive
    pred = p.argmax(axis=1)
    return pd.Series({"mood": (p[:, 2] - p[:, 0]).mean(),
                      "pos_share": (pred == 2).mean(),
                      "neg_share": (pred == 0).mean(),
                      "n": len(headlines)})

feats = []
for k, hs in enumerate(days["headlines"]):
    feats.append(day_features(hs))
    if k % 200 == 0:
        print(f"{k}/{len(days)} days done")
days = pd.concat([days, pd.DataFrame(feats)], axis=1)

OUT = PROJECT_ROOT / "results" / "nifty_daily_sentiment.csv"
days[["date", "split", "label", "pct_change", "mood", "pos_share", "neg_share", "n"]].to_csv(OUT, index=False)
print("saved", OUT)

KeyboardInterrupt: 

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

d = pd.read_csv("/content/drive/MyDrive/financial-news-intelligence/results/nifty_daily_sentiment.csv")
print(d.groupby("split")["n"].mean().round(0))

for s in ["train", "valid", "test"]:
    x = d[d.split == s]
    print(s, "correlation(mood, next-day change) =", round(x["mood"].corr(x["pct_change"]), 3))

F = ["mood", "pos_share", "neg_share"]
tr = d[d.split == "train"]
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw, max_iter=1000).fit(tr[F], tr["label"])
    for s in ["valid", "test"]:
        x = d[d.split == s]; p = m.predict(x[F])
        print(f"LR class_weight={cw} {s}: acc {accuracy_score(x.label, p):.3f}  Macro-F1 {f1_score(x.label, p, average='macro'):.3f}")
for s in ["valid", "test"]:
    x = d[d.split == s]; p = ["Neutral"] * len(x)
    print(f"Always-Neutral {s}: acc {accuracy_score(x.label, p):.3f}  Macro-F1 {f1_score(x.label, p, average='macro'):.3f}")

split
test      26.0
train    278.0
valid     54.0
Name: n, dtype: float64
train correlation(mood, next-day change) = 0.027
valid correlation(mood, next-day change) = -0.141
test correlation(mood, next-day change) = -0.11
LR class_weight=None valid: acc 0.596  Macro-F1 0.249
LR class_weight=None test: acc 0.451  Macro-F1 0.207
LR class_weight=balanced valid: acc 0.297  Macro-F1 0.284
LR class_weight=balanced test: acc 0.290  Macro-F1 0.251
Always-Neutral valid: acc 0.596  Macro-F1 0.249
Always-Neutral test: acc 0.451  Macro-F1 0.207
